# Delta Lake Assignment

## Objective

Perform Incremental Data Processing using Delta Lake.

### Dataset

Sample Superstore Dataset

### Tasks

- Load CSV file
- Clean Data
- Save as Delta Table
- Create Incremental Data
- Perform MERGE (UPSERT)
- Validate Results
- Display Final Output


#Import Libraries

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from delta.tables import DeltaTable

# Create Spark Session

In [0]:
spark = SparkSession.builder \
    .appName("Delta Lake Assignment") \
    .getOrCreate()

print("Spark Session Created Successfully")

Spark Session Created Successfully


#  Load the CSV Dataset

In [0]:
superstore_df = spark.read.csv(
    "/Volumes/my_databricks_workspace/default/samplesuperstore/Sample - Superstore.csv",
    header=True,
    inferSchema=True
)

display(superstore_df.limit(20))# Display only the first 20 rows of the dataset

Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2,0,41.9136
2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs, Rounded Back",731.94,3,0,219.582
3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters by Universal,14.62,2,0,6.8714
4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.031
5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.368,2,0.2,2.5164
6,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,FUR-FU-10001487,Furniture,Furnishings,"Eldon Expressions Wood and Plastic Desk Accessories, Cherry Wood",48.86,7,0,14.1694
7,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-AR-10002833,Office Supplies,Art,Newell 322,7.28,4,0,1.9656
8,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,TEC-PH-10002275,Technology,Phones,Mitel 5320 IP Phone VoIP phone,907.152,6,0.2,90.7152
9,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-BI-10003910,Office Supplies,Binders,DXL Angle-View Binders with Locking Rings by Samsill,18.504,3,0.2,5.7825
10,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-AP-10002892,Office Supplies,Appliances,Belkin F5C206VTEL 6 Outlet Surge,114.9,5,0,34.47


# Display Dataset Schema



In [0]:
# Display the schema of the DataFrame
superstore_df.printSchema()

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: date (nullable = true)
 |-- Ship Date: date (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Discount: string (nullable = true)
 |-- Profit: double (nullable = true)



# Count Total Records

In [0]:
# Count the total number of rows
total_records = superstore_df.count()

# Print the total number of records
print("Total Number of Records:", total_records)

Total Number of Records: 9994


# Check for Null Values

In [0]:
from pyspark.sql.functions import col, when, count

# Count null values in each column
null_values = superstore_df.select([
    count(when(col(column).isNull(), column)).alias(column)
    for column in superstore_df.columns
])

# Display null count for every column
display(null_values)

Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


# Remove Null Values

In [0]:
# Remove rows containing null values
clean_df = superstore_df.dropna()

# Display the first 20 rows after removing null values
display(clean_df.limit(20))

Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2,0,41.9136
2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs, Rounded Back",731.94,3,0,219.582
3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters by Universal,14.62,2,0,6.8714
4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.031
5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.368,2,0.2,2.5164
6,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,FUR-FU-10001487,Furniture,Furnishings,"Eldon Expressions Wood and Plastic Desk Accessories, Cherry Wood",48.86,7,0,14.1694
7,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-AR-10002833,Office Supplies,Art,Newell 322,7.28,4,0,1.9656
8,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,TEC-PH-10002275,Technology,Phones,Mitel 5320 IP Phone VoIP phone,907.152,6,0.2,90.7152
9,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-BI-10003910,Office Supplies,Binders,DXL Angle-View Binders with Locking Rings by Samsill,18.504,3,0.2,5.7825
10,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-AP-10002892,Office Supplies,Appliances,Belkin F5C206VTEL 6 Outlet Surge,114.9,5,0,34.47


In [0]:
# Count the number of records after removing null values
print("Records After Removing Null Values:", clean_df.count())

Records After Removing Null Values: 9994


# Remove Duplicate Records





In [0]:
# Remove duplicate rows
clean_df = clean_df.dropDuplicates()

# Display the cleaned dataset
display(clean_df.limit(20))

Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.368,2,0.2,2.5164
17,CA-2014-105893,2014-11-11,2014-11-18,Standard Class,PK-19075,Pete Kriz,Consumer,United States,Madison,Wisconsin,53711,Central,OFF-ST-10004186,Office Supplies,Storage,"""Stur-D-Stor Shelving, Vertical 5-Shelf: 72""""H x 36""""W x 18 1/2""""D""",665.88,6,0,13.3176
21,CA-2014-143336,2014-08-27,2014-09-01,Second Class,ZD-21925,Zuschuss Donatelli,Consumer,United States,San Francisco,California,94109,West,OFF-BI-10002215,Office Supplies,Binders,"Wilson Jones Hanging View Binder, White, 1""""",22.72,4,0.2,7.384
53,CA-2015-115742,2015-04-18,2015-04-22,Standard Class,DP-13000,Darren Powers,Consumer,United States,New Albany,Indiana,47150,Central,FUR-CH-10003061,Furniture,Chairs,"Global Leather Task Chair, Black",89.99,1,0,17.0981
118,CA-2015-110457,2015-03-02,2015-03-06,Standard Class,DK-13090,Dave Kipp,Consumer,United States,Seattle,Washington,98103,West,FUR-TA-10001768,Furniture,Tables,Hon Racetrack Conference Tables,787.53,3,0,165.3813
140,CA-2016-145583,2016-10-13,2016-10-19,Standard Class,LC-16885,Lena Creighton,Consumer,United States,Roseville,California,95661,West,FUR-FU-10001706,Furniture,Furnishings,Longer-Life Soft White Bulbs,43.12,14,0,20.6976
145,CA-2017-155376,2017-12-22,2017-12-27,Standard Class,SG-20080,Sandra Glassco,Consumer,United States,Independence,Missouri,64055,Central,OFF-AP-10001058,Office Supplies,Appliances,Sanyo 2.5 Cubic Foot Mid-Size Office Refrigerators,839.43,3,0,218.2518
147,CA-2014-110072,2014-10-22,2014-10-28,Standard Class,MG-17680,Maureen Gastineau,Home Office,United States,Newark,Ohio,43055,East,FUR-FU-10000521,Furniture,Furnishings,"""Seth Thomas 14"""" Putty-Colored Wall Clock""",93.888,4,0.2,12.9096
186,CA-2016-105018,2016-11-28,2016-12-02,Standard Class,SK-19990,Sally Knutson,Consumer,United States,Fairfield,Connecticut,6824,East,OFF-BI-10001890,Office Supplies,Binders,Avery Poly Binder Pockets,7.16,2,0,3.4368
200,US-2017-124303,2017-07-06,2017-07-13,Standard Class,FH-14365,Fred Hopkins,Corporate,United States,Philadelphia,Pennsylvania,19120,East,OFF-PA-10002749,Office Supplies,Paper,"Wirebound Message Books, 5-1/2 x 4 Forms, 2 or 4 Forms per Page",16.056,3,0.2,5.8203


## Verify Record Count

In [0]:
# Count the records after removing duplicates
print("Records After Removing Duplicates:", clean_df.count())

Records After Removing Duplicates: 9994


# Rename Column Names




In [0]:
# Replace spaces in column names with underscores

clean_df = clean_df.toDF(*[
    column.replace(" ", "_")
    for column in clean_df.columns
])

# Display the updated column names
print(clean_df.columns)

['Row_ID', 'Order_ID', 'Order_Date', 'Ship_Date', 'Ship_Mode', 'Customer_ID', 'Customer_Name', 'Segment', 'Country', 'City', 'State', 'Postal_Code', 'Region', 'Product_ID', 'Category', 'Sub-Category', 'Product_Name', 'Sales', 'Quantity', 'Discount', 'Profit']


# Save the Cleaned Data as a Delta Table



In [0]:
# Save the cleaned DataFrame as a Delta table
clean_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/Volumes/my_databricks_workspace/default/samplesuperstore/superstore_delta")

print("Delta Table Created Successfully.")

Delta Table Created Successfully.


#  Read the Delta Table



In [0]:
# Read the Delta table

delta_df = spark.read \
    .format("delta") \
    .load("/Volumes/my_databricks_workspace/default/samplesuperstore/superstore_delta")

display(delta_df.limit(20))

Row_ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer_Name,Segment,Country,City,State,Postal_Code,Region,Product_ID,Category,Sub-Category,Product_Name,Sales,Quantity,Discount,Profit
5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.368,2,0.2,2.5164
17,CA-2014-105893,2014-11-11,2014-11-18,Standard Class,PK-19075,Pete Kriz,Consumer,United States,Madison,Wisconsin,53711,Central,OFF-ST-10004186,Office Supplies,Storage,"""Stur-D-Stor Shelving, Vertical 5-Shelf: 72""""H x 36""""W x 18 1/2""""D""",665.88,6,0,13.3176
21,CA-2014-143336,2014-08-27,2014-09-01,Second Class,ZD-21925,Zuschuss Donatelli,Consumer,United States,San Francisco,California,94109,West,OFF-BI-10002215,Office Supplies,Binders,"Wilson Jones Hanging View Binder, White, 1""""",22.72,4,0.2,7.384
53,CA-2015-115742,2015-04-18,2015-04-22,Standard Class,DP-13000,Darren Powers,Consumer,United States,New Albany,Indiana,47150,Central,FUR-CH-10003061,Furniture,Chairs,"Global Leather Task Chair, Black",89.99,1,0,17.0981
118,CA-2015-110457,2015-03-02,2015-03-06,Standard Class,DK-13090,Dave Kipp,Consumer,United States,Seattle,Washington,98103,West,FUR-TA-10001768,Furniture,Tables,Hon Racetrack Conference Tables,787.53,3,0,165.3813
140,CA-2016-145583,2016-10-13,2016-10-19,Standard Class,LC-16885,Lena Creighton,Consumer,United States,Roseville,California,95661,West,FUR-FU-10001706,Furniture,Furnishings,Longer-Life Soft White Bulbs,43.12,14,0,20.6976
145,CA-2017-155376,2017-12-22,2017-12-27,Standard Class,SG-20080,Sandra Glassco,Consumer,United States,Independence,Missouri,64055,Central,OFF-AP-10001058,Office Supplies,Appliances,Sanyo 2.5 Cubic Foot Mid-Size Office Refrigerators,839.43,3,0,218.2518
147,CA-2014-110072,2014-10-22,2014-10-28,Standard Class,MG-17680,Maureen Gastineau,Home Office,United States,Newark,Ohio,43055,East,FUR-FU-10000521,Furniture,Furnishings,"""Seth Thomas 14"""" Putty-Colored Wall Clock""",93.888,4,0.2,12.9096
186,CA-2016-105018,2016-11-28,2016-12-02,Standard Class,SK-19990,Sally Knutson,Consumer,United States,Fairfield,Connecticut,6824,East,OFF-BI-10001890,Office Supplies,Binders,Avery Poly Binder Pockets,7.16,2,0,3.4368
200,US-2017-124303,2017-07-06,2017-07-13,Standard Class,FH-14365,Fred Hopkins,Corporate,United States,Philadelphia,Pennsylvania,19120,East,OFF-PA-10002749,Office Supplies,Paper,"Wirebound Message Books, 5-1/2 x 4 Forms, 2 or 4 Forms per Page",16.056,3,0.2,5.8203


# Verify Total Records

In [0]:
# Count the total number of records in the Delta table

print("Total Records in Delta Table:", delta_df.count())

Total Records in Delta Table: 9994


# Create an Incremental Dataset


## Objective
Create a second dataset to simulate new incoming records.

## Description
In real-world scenarios, new data is continuously received. Instead of loading the entire dataset again, only new and updated records are processed.

For this assignment:

- Existing records will be **updated**.
- New records will be **inserted**.

This process is called **Incremental Data Processing**.

#Load the Incremental Dataset

In [0]:
# Read the incremental CSV file
increment_df = spark.read.csv(
    "/Volumes/my_databricks_workspace/default/samplesuperstore/Superstore_Incremental.csv",
    header=True,
    inferSchema=True
)

# Display the first 20 rows
display(increment_df.limit(20))

Row_ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer_Name,Segment,Country,City,State,Postal_Code,Region,Product_ID,Category,Sub-Category,Product_Name,Sales,Quantity,Discount,Profit
2,CA-2016-152156,11/8/2016,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,Hon Deluxe Fabric Upholstered Stacking Chairs,750,3,0.0,225
9995,CA-2018-200001,1/5/2018,2018-01-09,Standard Class,AB-11111,John Smith,Consumer,United States,Chicago,Illinois,60601,Central,OFF-BI-10000001,Office Supplies,Binders,Premium Binder,350,2,0.1,45
9996,CA-2018-200002,2/10/2018,2018-02-15,First Class,CD-22222,Alice Brown,Corporate,United States,Dallas,Texas,75001,Central,TEC-PH-10000002,Technology,Phones,Wireless Phone,650,5,0.0,150


## Check the number of records

In [0]:
print("Incremental Records:", increment_df.count())

Incremental Records: 3


# Create a DeltaTable Object

Create a DeltaTable object to perform the MERGE operation.


The DeltaTable API allows us to update, insert, and delete records in a Delta table. It is required before performing the MERGE (UPSERT) operation.

In [0]:
from delta.tables import DeltaTable

# Create a DeltaTable object from the saved Delta table
deltaTable = DeltaTable.forPath(
    spark,
    "/Volumes/my_databricks_workspace/default/samplesuperstore/superstore_delta"
)

print("DeltaTable Object Created Successfully.")

DeltaTable Object Created Successfully.


#  Perform the MERGE (UPSERT) Operation


Merge the incremental dataset into the existing Delta table.

The MERGE operation performs two actions:

- **Update** existing records when the Row_ID already exists.
- **Insert** new records when the Row_ID does not exist.

This enables efficient incremental data processing without reloading the full dataset.

In [0]:
deltaTable.alias("old") \
.merge(
    increment_df.alias("new"),
    "old.Row_ID = new.Row_ID"
) \
.whenMatchedUpdateAll() \
.whenNotMatchedInsertAll() \
.execute()

print("MERGE Operation Completed Successfully.")

MERGE Operation Completed Successfully.


#  Display the Updated Delta Table

Display the Delta table after performing the MERGE operation.


In [0]:
# Read the updated Delta table

final_df = spark.read \
    .format("delta") \
    .load("/Volumes/my_databricks_workspace/default/samplesuperstore/superstore_delta")

# Display the first 20 records
display(final_df.limit(20))

Row_ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer_Name,Segment,Country,City,State,Postal_Code,Region,Product_ID,Category,Sub-Category,Product_Name,Sales,Quantity,Discount,Profit
5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.368,2,0.2,2.5164
17,CA-2014-105893,2014-11-11,2014-11-18,Standard Class,PK-19075,Pete Kriz,Consumer,United States,Madison,Wisconsin,53711,Central,OFF-ST-10004186,Office Supplies,Storage,"""Stur-D-Stor Shelving, Vertical 5-Shelf: 72""""H x 36""""W x 18 1/2""""D""",665.88,6,0,13.3176
21,CA-2014-143336,2014-08-27,2014-09-01,Second Class,ZD-21925,Zuschuss Donatelli,Consumer,United States,San Francisco,California,94109,West,OFF-BI-10002215,Office Supplies,Binders,"Wilson Jones Hanging View Binder, White, 1""""",22.72,4,0.2,7.384
53,CA-2015-115742,2015-04-18,2015-04-22,Standard Class,DP-13000,Darren Powers,Consumer,United States,New Albany,Indiana,47150,Central,FUR-CH-10003061,Furniture,Chairs,"Global Leather Task Chair, Black",89.99,1,0,17.0981
118,CA-2015-110457,2015-03-02,2015-03-06,Standard Class,DK-13090,Dave Kipp,Consumer,United States,Seattle,Washington,98103,West,FUR-TA-10001768,Furniture,Tables,Hon Racetrack Conference Tables,787.53,3,0,165.3813
140,CA-2016-145583,2016-10-13,2016-10-19,Standard Class,LC-16885,Lena Creighton,Consumer,United States,Roseville,California,95661,West,FUR-FU-10001706,Furniture,Furnishings,Longer-Life Soft White Bulbs,43.12,14,0,20.6976
145,CA-2017-155376,2017-12-22,2017-12-27,Standard Class,SG-20080,Sandra Glassco,Consumer,United States,Independence,Missouri,64055,Central,OFF-AP-10001058,Office Supplies,Appliances,Sanyo 2.5 Cubic Foot Mid-Size Office Refrigerators,839.43,3,0,218.2518
147,CA-2014-110072,2014-10-22,2014-10-28,Standard Class,MG-17680,Maureen Gastineau,Home Office,United States,Newark,Ohio,43055,East,FUR-FU-10000521,Furniture,Furnishings,"""Seth Thomas 14"""" Putty-Colored Wall Clock""",93.888,4,0.2,12.9096
186,CA-2016-105018,2016-11-28,2016-12-02,Standard Class,SK-19990,Sally Knutson,Consumer,United States,Fairfield,Connecticut,6824,East,OFF-BI-10001890,Office Supplies,Binders,Avery Poly Binder Pockets,7.16,2,0,3.4368
200,US-2017-124303,2017-07-06,2017-07-13,Standard Class,FH-14365,Fred Hopkins,Corporate,United States,Philadelphia,Pennsylvania,19120,East,OFF-PA-10002749,Office Supplies,Paper,"Wirebound Message Books, 5-1/2 x 4 Forms, 2 or 4 Forms per Page",16.056,3,0.2,5.8203


## Validate the Row Count



In [0]:
# Count the total number of records after MERGE
row_count = final_df.count()

print("Total Records After MERGE:", row_count)

Total Records After MERGE: 9996


#  Check for Duplicate Records


Ensure that there are no duplicate Row_ID values after the MERGE operation.


In [0]:
# Find duplicate Row_ID values
duplicates = final_df.groupBy("Row_ID") \
    .count() \
    .filter("count > 1")

display(duplicates)

Row_ID,count


#  Generate Summary Statistics

Display summary statistics for numerical columns.

The `describe()` function provides useful statistics such as count, mean, standard deviation, minimum, and maximum values for the numerical columns in the dataset.

In [0]:
# Display summary statistics
final_df.describe().show()

+-------+-----------------+--------------+--------------+-----------+------------------+-----------+-------------+--------+-------+------------------+-------+---------------+----------+------------+--------------------+------------------+------------------+------------------+------------------+
|summary|           Row_ID|      Order_ID|     Ship_Mode|Customer_ID|     Customer_Name|    Segment|      Country|    City|  State|       Postal_Code| Region|     Product_ID|  Category|Sub-Category|        Product_Name|             Sales|          Quantity|          Discount|            Profit|
+-------+-----------------+--------------+--------------+-----------+------------------+-----------+-------------+--------+-------+------------------+-------+---------------+----------+------------+--------------------+------------------+------------------+------------------+------------------+
|  count|             9996|          9996|          9996|       9996|              9996|       9996|         999

#  Display the Final Dataset


Display the final Delta table after all processing is complete.

This is the final output of the assignment, showing the dataset after cleaning, Delta conversion, and incremental processing using the MERGE operation.

In [0]:
# Display the first 20 rows of the final dataset
display(final_df.limit(20))

Row_ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer_Name,Segment,Country,City,State,Postal_Code,Region,Product_ID,Category,Sub-Category,Product_Name,Sales,Quantity,Discount,Profit
5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.368,2,0.2,2.5164
17,CA-2014-105893,2014-11-11,2014-11-18,Standard Class,PK-19075,Pete Kriz,Consumer,United States,Madison,Wisconsin,53711,Central,OFF-ST-10004186,Office Supplies,Storage,"""Stur-D-Stor Shelving, Vertical 5-Shelf: 72""""H x 36""""W x 18 1/2""""D""",665.88,6,0,13.3176
21,CA-2014-143336,2014-08-27,2014-09-01,Second Class,ZD-21925,Zuschuss Donatelli,Consumer,United States,San Francisco,California,94109,West,OFF-BI-10002215,Office Supplies,Binders,"Wilson Jones Hanging View Binder, White, 1""""",22.72,4,0.2,7.384
53,CA-2015-115742,2015-04-18,2015-04-22,Standard Class,DP-13000,Darren Powers,Consumer,United States,New Albany,Indiana,47150,Central,FUR-CH-10003061,Furniture,Chairs,"Global Leather Task Chair, Black",89.99,1,0,17.0981
118,CA-2015-110457,2015-03-02,2015-03-06,Standard Class,DK-13090,Dave Kipp,Consumer,United States,Seattle,Washington,98103,West,FUR-TA-10001768,Furniture,Tables,Hon Racetrack Conference Tables,787.53,3,0,165.3813
140,CA-2016-145583,2016-10-13,2016-10-19,Standard Class,LC-16885,Lena Creighton,Consumer,United States,Roseville,California,95661,West,FUR-FU-10001706,Furniture,Furnishings,Longer-Life Soft White Bulbs,43.12,14,0,20.6976
145,CA-2017-155376,2017-12-22,2017-12-27,Standard Class,SG-20080,Sandra Glassco,Consumer,United States,Independence,Missouri,64055,Central,OFF-AP-10001058,Office Supplies,Appliances,Sanyo 2.5 Cubic Foot Mid-Size Office Refrigerators,839.43,3,0,218.2518
147,CA-2014-110072,2014-10-22,2014-10-28,Standard Class,MG-17680,Maureen Gastineau,Home Office,United States,Newark,Ohio,43055,East,FUR-FU-10000521,Furniture,Furnishings,"""Seth Thomas 14"""" Putty-Colored Wall Clock""",93.888,4,0.2,12.9096
186,CA-2016-105018,2016-11-28,2016-12-02,Standard Class,SK-19990,Sally Knutson,Consumer,United States,Fairfield,Connecticut,6824,East,OFF-BI-10001890,Office Supplies,Binders,Avery Poly Binder Pockets,7.16,2,0,3.4368
200,US-2017-124303,2017-07-06,2017-07-13,Standard Class,FH-14365,Fred Hopkins,Corporate,United States,Philadelphia,Pennsylvania,19120,East,OFF-PA-10002749,Office Supplies,Paper,"Wirebound Message Books, 5-1/2 x 4 Forms, 2 or 4 Forms per Page",16.056,3,0.2,5.8203


# Assignment Summary

## Objective Achieved

The objective of this assignment was to perform incremental data processing using Delta Lake.

### Tasks Completed

- Loaded the Sample Superstore dataset into a Spark DataFrame.
- Cleaned the dataset by removing null values and duplicate records.
- Renamed column names to make them compatible with Delta Lake.
- Stored the cleaned dataset as a Delta table.
- Created and loaded an incremental dataset.
- Performed a MERGE (UPSERT) operation to update existing records and insert new records.
- Validated the updated data by checking row count and duplicate records.
- Generated summary statistics and displayed the final dataset.

## Conclusion

Delta Lake successfully handled incremental data processing using the MERGE operation. Existing records were updated, new records were inserted, and data consistency was maintained through Delta Lake's ACID transaction support.